# AI Engineering Challenge — Local RAG Question Answering

This notebook already runs end-to-end and produces an answer to every question — it's just **bad**.
Your job: make it better. Improve the score, submit early, keep iterating.

**Rules**
- Edit anything in this notebook or in `rag/` freely.
- Do **not** edit anything in `eval/` — that's the locked scoring harness.
- Score by running the last cell as often as you like.


## Setup — locate the project root and load the corpus

In [ ]:
import sys, pathlib, json

def find_root(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("Could not find project root (pyproject.toml not found)")

ROOT = find_root(pathlib.Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag.llm import call_llm, PROVIDER, DEFAULT_MODEL

CORPUS_PATH = ROOT / "data" / "corpus.json"
DEV_QA_PATH = ROOT / "data" / "dev_qa.json"
TEST_QA_PATH = ROOT / "eval" / "test_qa.json"

if CORPUS_PATH.exists() and DEV_QA_PATH.exists() and TEST_QA_PATH.exists():
    corpus = json.loads(CORPUS_PATH.read_text())
    dev_qa = json.loads(DEV_QA_PATH.read_text())
else:
    # First run: no local dataset yet (it's gitignored, never pushed to the repo).
    # Build it from SQuAD and cache it locally so later runs are instant. The split
    # is deterministic (fixed seed), so eval/test_qa.json comes out identical for
    # every participant without ever being committed either.
    print("No local dataset found — downloading SQuAD and building the corpus (one-time, needs internet)...")
    from rag.dataset import build_splits

    corpus, dev_qa, test_qa = build_splits()
    CORPUS_PATH.parent.mkdir(exist_ok=True)
    TEST_QA_PATH.parent.mkdir(exist_ok=True)
    CORPUS_PATH.write_text(json.dumps(corpus, indent=2))
    DEV_QA_PATH.write_text(json.dumps(dev_qa, indent=2))
    TEST_QA_PATH.write_text(json.dumps(test_qa, indent=2))

print(f"LLM provider: {PROVIDER}  |  model: {DEFAULT_MODEL}")
print(f"Corpus: {len(corpus)} articles, {sum(len(d['text']) for d in corpus):,} chars total")
print(f"Dev QA: {len(dev_qa)} questions (visible, gold answers included)")


## Step 1 — Retrieval (baseline, intentionally crude)

The baseline chunks every article into fixed 500-character slices (ignoring word/sentence
boundaries), represents each chunk as **raw word counts** (no normalization, no IDF), and
ranks chunks by plain dot-product against the question's raw count vector. It works, but
common words dominate the ranking and chunk boundaries cut sentences in half.


In [6]:
import numpy as np
import re

CHUNK_SIZE = 500  # crude: fixed character window, no overlap, no sentence awareness

def chunk_text(doc_id: str, text: str, chunk_size: int = CHUNK_SIZE) -> list[dict]:
    return [
        {"doc_id": doc_id, "text": text[i : i + chunk_size]}
        for i in range(0, len(text), chunk_size)
    ]

chunks = [c for doc in corpus for c in chunk_text(doc["doc_id"], doc["text"])]
print(f"{len(chunks)} chunks built from {len(corpus)} articles")

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

vocab = {}
for c in chunks:
    for tok in tokenize(c["text"]):
        vocab.setdefault(tok, len(vocab))

def vectorize(text: str) -> np.ndarray:
    vec = np.zeros(len(vocab), dtype=np.float32)
    for tok in tokenize(text):
        idx = vocab.get(tok)
        if idx is not None:
            vec[idx] += 1.0  # raw counts -- no normalization, no IDF weighting
    return vec

chunk_vectors = np.stack([vectorize(c["text"]) for c in chunks])
print(f"vocab size: {len(vocab)}  |  chunk matrix: {chunk_vectors.shape}")

def retrieve(question: str, k: int = 1) -> list[dict]:
    q_vec = vectorize(question)
    scores = chunk_vectors @ q_vec  # plain dot product
    top_idx = np.argsort(-scores)[:k]
    return [chunks[i] for i in top_idx]


1320 chunks built from 20 articles
vocab size: 14298  |  chunk matrix: (1320, 14298)


## Step 2 — Generate an answer from the retrieved context

Single retrieved chunk, no reranking, a bare-bones prompt.


In [10]:
PROMPT_TEMPLATE = """Answer the question using only the context below. \
If the answer isn't in the context, say so briefly.

Context:
{context}

Question: {question}

Answer:"""

def answer_question(question: str) -> dict:
    retrieved = retrieve(question, k=1)
    context = "\n\n".join(r["text"] for r in retrieved)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    answer = call_llm(prompt)
    citations = list({r["doc_id"] for r in retrieved})
    return {"answer": answer.strip(), "citations": citations}

# quick smoke test
sample = dev_qa[0]
result = answer_question(sample["question"])
print("Q:", sample["question"])
print("gold:", sample["answers"])
print("predicted:", result)


Q: Who conceptualized the piston?
gold: ['Papin']
predicted: {'answer': 'There is no mention of the piston in the provided context. The context appears to be related to the geologic time scale, the Rhine river, and its history, but it does not mention the piston.', 'citations': ['Rhine', 'Geology']}


## Step 3 — Self-check on the dev set (visible, iterate freely against this)

This is *not* the score that counts — it's here so you can debug and see gold answers.
The real score comes from the hidden set in the next section.


In [8]:
from eval.metrics import exact_match_score, f1_score

f1_total = 0.0
for item in dev_qa:
    result = answer_question(item["question"])
    f1 = f1_score(result["answer"], item["answers"])
    f1_total += f1

print(f"dev F1 (informal, {len(dev_qa)} questions): {f1_total / len(dev_qa):.3f}")


dev F1 (informal, 40 questions): 0.105


## Step 4 — Submit: score against the hidden test set

Run this whenever you want an official score. **Do not edit `eval/`.**
Raise a green post-it and an organizer will log the printed `score`.


In [11]:
from eval.harness import run_hidden_eval

results = run_hidden_eval(answer_question)
results


  scored 10/80...
  scored 20/80...
  scored 30/80...
  scored 40/80...
  scored 50/80...
  scored 60/80...
  scored 70/80...
  scored 80/80...

=== RESULT ===
  score (higher is better)        : 26.26
  total_tokens (lower is more frugal): 52107
  ---
  n_questions                     : 80
  exact_match                     : 0.1
  f1                              : 0.1751
  citation_hit_rate               : 0.6125
  errors                          : 0
  total_time_sec                  : 429.2
  avg_latency_sec                 : 5.36
  input_tokens                    : 50188
  output_tokens                   : 1919
  avg_tokens_per_question         : 651.3


{'n_questions': 80,
 'exact_match': 0.1,
 'f1': 0.1751,
 'citation_hit_rate': 0.6125,
 'score': 26.26,
 'errors': 0,
 'total_time_sec': 429.2,
 'avg_latency_sec': 5.36,
 'total_tokens': 52107,
 'input_tokens': 50188,
 'output_tokens': 1919,
 'avg_tokens_per_question': 651.3}

Both count toward the accuracy score: `score = 0.8 * answer_f1 + 0.2 * citation_hit_rate`.
Getting the right document isn't enough if the extracted answer text is wrong, and
vice versa.

There's a second, independent score too: `total_tokens` (lower is more frugal),
tracked automatically for every call through `call_llm`. A submission that gets
90% of the accuracy for a fraction of the tokens (e.g. shorter prompts, less
retrieved context, a smaller model) is a legitimate way to win on that axis even
if it doesn't top the accuracy leaderboard.
